# သင်ခန်းစာ 12 - အသံပြောအတိတ်မှတ်တမ်း လျှော့ချခြင်းနှင့် Agent Scratchpad

ဤ notebook သည် Microsoft Agent Framework ကို အသုံးပြု၍ အခြေအနေကို မိမိထိန်းသိမ်းနိုင်စွမ်းကို ရှင်းပြပေးသည်။ စကားပြောကြားမှုများတိုးလာသည်နှင့် အတူ token အရေအတွက်တိုးလာပြီး — နောက်ဆုံးတွင် မော်ဒယ်၏ အခြေအနေ ပြခန်းကို ကျော်လွန်သွားသည်။ ဤပြဿနာအား **အခြေအနေ စုစုပေါင်း နည်းလမ်း** နှင့် **agent scratchpad** တို့ဖြင့် ဖြေရှင်းသည်။

## သင်ယူမည့်အရာများ:
1. **အခြေအနေ စီမံခန့်ခွဲမှု အရေးကြီးသည့် အကြောင်း**: token ကန့်သတ်ချက်များနှင့် အခြေအနေ ပြခန်းများကို နားလည်ခြင်း
2. **အခြေအနေ အသိပညာရှိသော Agents**: သူတို့တို့ စကားဝိုင်း အခြေအနေကို ကိုယ်တိုင်စီမံနိုင်သော agent များ တည်ဆောက်ခြင်း
3. **အခြေအနေ စုစုပေါင်း နည်းလမ်း**: စကားပြောကြားမှတ်တမ်းကို တိုတောင်းစေသော ကိရိယာများကို အသုံးပြုခြင်း
4. **Agent Scratchpad**: အခြေအနေ လျှော့ချခြင်းကို ကျော်လွန်သွားသော အမြဲတမ်းမှတ်ဉာဏ်

## လိုအပ်သော အချက်များ:
- Azure OpenAI ကို ပတ်ဝန်းကျင်အတန်းသတ်မှတ်ချက်များဖြင့် ပြင်ဆင်ထားရန်
- ယခင် သင်ခန်းစာများမှ မူလ agent အကြောင်း အခြေခံကို နားလည်ထားခြင်း


## ဆက်တင်တပ်ဆင်ခြင်း


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## ဘာကြောင့် Context စီမံခန့်ခွဲမှုအရေးကြီးသလဲ

လုပ်ဆောင်ချက်မှတ်တမ်းတစ်ခုစီတွင် LLM တစ်ခုစီမှာ **context window** ကန့်သတ်ချက်ရှိသည် — တစ်ကြိမ်တည်းတောင်းဆိုမှုတစ်ခုအတွင်း အများဆုံး token အရေအတွက်ဖြစ်သည်။ မပြတ်တောက်ပြောဆိုမှုတစ်ခု တိုးတက်လာသည့်အခါတွင် -

- **Token အရေအတွက်သည် အသုံးပြုသူစာတိုက်နှင့် အကူအညီပေးသူတုံ့ပြန်ချက်တိုင်းတွင် လိုင်းယားတိုးတက်လျက်ရှိသည်။**
- **Prompt token များသည် ကုန်ကျစရိတ် အများဆုံဖြစ်သည်** ဘာကြောင့်ဆိုသော် လုံးဝ သမိုင်းကြောင်းကို တစ်ခါ တင်ပို့ပေသည်။
- နောက်ဆုံးတွင် ပြောဆိုမှုသည် **context window ထက်ကျော်လွန်သွား၍** မော်ဒယ်သည်ဖြတ်တောက်ခြင်း သို့မဟုတ် အမှားထွက်ခြင်း ဖြစ်ပေါ်တတ်သည်။

### Context ကို စီမံခန့်ခွဲရန် မဟာဗျူဟာများ

| မဟာဗျူဟာ | ဖော်ပြချက် | အပြန်အလှန် နည်းလမ်း |
|---|---|---|
| **ဖြတ်တောက်ခြင်း** | အဟောင်းဆုံးစာတိုက်များကို ဖယ်ရှားသည် | မူလ context ကိုပျောက်ဆုံးစေသည် |
| **အကျဉ်းချုပ်ရေးခြင်း** | အဟောင်းစာတိုက်များကို အကျဉ်းချုပ်ထဲသို့ တုပ်ဆိုသည် | အသေးစိတ်အချို့ ပျောက်ဆုံးသော်လည်း အဓိကအချက်များ ထိန်းသိမ်းသည် |
| **Scratchpad / ပြင်ပ မှတ္ဉာဏ်** | ပြောဆိုမှုအပြင် key ဖြစ်သောအချက်များကို သိမ်းဆည်းသည် | ကိရိယာခေါ်ဆိုမှုလိုအပ်သည်၊ ဒါပေမယ့် လျှော့ချမှုများကို ပြောင်းလဲကိုင်တွယ်နိုင်သည် |

ဤ notebook တွင် ကျွန်ုပ်တို့သည် **အကျဉ်းချုပ်ခြင်း** နှင့် **scratchpad ကိရိယာ** ကို ပေါင်းစပ်သုံးပြီး စကားဝိုင်းသမိုင်း ကြေညာမှုကို သက်ဆိုင်ရာနေရာသို့ထားသောအချိန်တွင် လက်ရှိစွမ်းဆောင်ရည်ကို ထိန်းသိမ်းနိုင်သည်။


## စနစ်အကြောင်းအရာသိရှိသော Agent တစ်ယောက် ဖန်တီးခြင်း


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## တိုးချဲ့သော စကားပြောပွဲတစ်ခု အတုယူခြင်း

ဖြစ်ပျက်မှုများနှင့်အတူ စကားပြောပွဲစဉ်တစ်ခုကို ခြေလှမ်းများဖြင့် လေ့လာကြည့်ကြရအောင်။ အေဂျင့်သည် အရေးပါတဲ့ အသေးစိတ်များ (နှစ်သက်မှုများ၊ ဘတ်ဂျက်၊ ခရီးသွားရက်များ) ကို အစဉ်အဆက် ထိန်းသိမ်းသင့်ပြီး ဆက်လက်မှုကို ပြသနိုင်ရမည်။


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

ဂျာပန်နိုင်ငံ၊ ဆူရှီ၊ ဘုရားတော်များ၊ ဓာတ်ပုံဖမ်းခြင်း၊ ၃၀၀၀ ဒေါ်လာစီမံကိန်း၊ တစ်ကိုယ်တည်း ခရီးသွားခြင်းနှင့် ဧပြီလအချိန်ကာလအကြောင်းများအား ဧည့်ခံသူက အစပိုင်းပတ်လည်များမှ ဂရုစိုက်ထားမှုကို သတိပြုပါ — အဆိုပါ ပြောဆိုချက်တိုတိုတွင် အလုပ်အဆင်ပြေသော်လည်း၊ စကားပြောဆိုမှုကြီးထွားလာသည်နှင့်အမျှ ပြီးခဲ့သည့်သမိုင်းမှတ်တမ်းများအား ပြန်ပို့ပေးရခြင်းသည် ကုန်ကျစရိတ်များ ဖြစ်လာသည်။

စကားပြောဆိုမှုကို ပိုမိုခေါက်ခွဲ၍ အဓိပ္ပာယ် စုဆောင်းမှုကို ကြည့်ရှုကြပါစို့-


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## အကြောင်းအရာ ချုပ်ထုတ်မှုပုံစံ

စကားပြောဆိုမှု တိုးတက်လာသည့်အခါ၊ ကျွန်ုပ်တို့သည် တစ်စုတစ်စည်း စုစည်းထားသော အကြောင်းအရာကို တစ်ကျော့ချုပ်ထုတ်ပေးနိုင်သော **ချုပ်ထုတ်မှုကိရိယာ** ကို အသုံးပြုနိုင်သည်။ အဲဒီကိရိယာကို စက်ကိုယ်ပြုသူက ခေါ်ယူပြီး အရေးကြီး ကြိုက်နှစ်သက်မှုများကို မှတ်တမ်းတင်ရန် အသုံးပြုသည်၊ ဒါ့ကြောင့် အဟောင်းဆောင်းသော စာတိုက်ချက်များကို ဖယ်ရှားပစ်ခဲ့သော်လည်း အရေးကြီးသော အချက်အလက်များကို ထိန်းသိမ်းထားနိုင်သည်။

ဤပုံစံသည် ပိုမိုတီထွင်ပြီး သမိုင်းကြောင်း လျှော့ချပေးခြင်းအတွက် အခြေခံအဆောက်အအုံဖြစ်သည်။
1. စက်ကိုယ်ပြုသူသည် စကားပြောဆိုမှုမှ အရေးကြီးသော အချက်အလက်များကို ရှာဖွေသည်
2. ထိုအချက်များကို ချုပ်ထုတ်မှုကိရိယာကို ခေါ်ယူ၍ သိုလှောင်သည်
3. ရှေးဟောင်းသော စာတိုက်ချက်များကို လုံခြုံစွာ ဖယ်ရှားနိုင်သည်၊ ချုပ်ထုတ်ချက်က အရေးကြီးသည့်အချက်များကို ဖော်ပြသောကြောင့် ဖြစ်သည်

အောက်တွင် စက်ကိုယ်ပြုသူက မိမိသိရှိလာသည့် အချက်အလက်များကို တစ်ကျော့ချုပ်ထုတ်၍ မှတ်တမ်းတင်နိုင်ရန် `summarize_preferences` ကိရိယာ တစ်ခုကို သတ်မှတ်ထားပါသည်။


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## အကျဉ်းချုပ်

ဒီသင်ခန်းစာမှာ Microsoft Agent Framework အသုံးပြုပြီး ရေရှည် agent စကားပြောပွဲတွေမှာ context ကို မည်သို့ စီမံခန့်ခွဲရမယ်ဆိုတာ လေ့လာသင်ကြားခဲ့သည်။

### အဓိကအတွေးအခေါ်များ
- **Context ပြတင်းပေါက်တွေမှာ ကန့်သတ်ချက်ရှိသေးတယ်** — စကားပြောမှတ်တမ်းထဲရှိ token တစ်ခုချင်းစီမှာ ကုန်ကျစရိတ်ဖြစ်ပြီး ကန့်သတ်ချက်ထဲမှာ ပါဝင်သည်။
- **အနှစ်ချုပ်ကိရိယာများ** က agent ကို စုပေါင်းထားသော context ကို တိုတိုချုပ်ချုပ် စာတိုစာညွှတ်အဖြစ် ပြောင်းလဲပေးပြီး token အသုံးပြုမူကို လျှော့ချပေးသည်။ သို့သော် အရေးကြီးသော သတင်းအချက်အလက်များကို ထိန်းသိမ်းထားသည်။
- **Agent scratchpads** က စကားပြောပွဲကို လျှော့ချမှုရှိသော်လည်း ရှိနေပြန်နိုင်သော ပဲ့တင်မှတ်ဉာဏ်ကို ပေးအပ်သည်။

### သင်တည်ဆောက်ခဲ့သည့် အရာများ
- အဆက်မပြတ်စကားပြောပွဲများအတွင်း ဆက်လက်သိမ်းဆည်းသည့် **context-aware agent** တစ်ခု
- အသုံးပြုသူအချက်အလက် အဓိကများကို အနှစ်ချုပ်ထားသော ပုံစံဖြင့် မှတ်တမ်းတင်သည့် **အနှစ်ချုပ်ကိရိယာ** (`summarize_preferences`)
- context ကို ထိန်းသိမ်းမှုနှင့် ပြောင်းလဲမှုကို ပြသသော **အဆက်မပြတ် စကားပြောပွဲ** တစ်ခု

### လက်တွေ့ အသုံးချမှုများ
- **ဖောက်သည်ဝန်ဆောင်မှု bot များ**: ရေရှည် ထောက်ပံ့မှု အစီအစဉ်များအတွင်း ဦးစားပေးချက်များကို မှတ်မိထားရန်
- **ကိုယ်ပိုင်အကူအညီပေးသူများ**: context ပြန်ရှင်းပြစရာမလိုဘဲ လက်ရှိစီမံကိန်းများကို ထိန်းသိမ်းရန်
- **ပညာရေးဆရာများ**: ကျောင်းသား များ၏ ပြီးပြည့်စုံမှုကို Interaction များစွာအတွင်း ထိန်းသိမ်းရန်

### နောက်တစ်ဆင့်လုပ်ဆောင်ရန်
- ဖိုင်အခြေပြု ထိန်းသိမ်းမှုဖြင့် ကမ္ဘာကြီးပြင် scratchpad ကိရိယာ တစ်ခု ပြုလုပ်ရန်
- အနှစ်ချုပ်ပြီးနောက် သမိုင်း စာရင်းများကို အလိုအလျောက် လျှော့ချမှု ထည့်သွင်းရန်
- semantic memory သွားပြန်ရှာဖွေမှုအတွက် vector databases နှင့် ပေါင်းစပ်ရန်
- အပြည့်အဝ context ဖြင့် ရက်ပေါင်းများစွာပြီးနောက် ဒါရိုက်တာများစကားပြန် ဆက်လုပ်နိုင်အောင် တည်ဆောက်ရန်


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ပြောကြားချက်**
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) အသုံးပြု၍ ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှန်ကန်မှုအတွက် ကြိုးပမ်းနေသော်လည်း၊ စက်ကိရိယာဘာသာပြန်ခြင်းများတွင် အမှားများ သို့မဟုတ် မှားယွင်းချက်များ ပါဝင်နိုင်ကြောင်း သတိပြုပါရန် လိုအပ်ပါသည်။ မူလစာတမ်းကို မူရင်းဘာသာဖြင့်သာ ယုံကြည်စိတ်ချရသော အချက်အလက်အဖြစ် သတ်မှတ်သင့်သည်။ အရေးကြီးသည့် သတင်းအချက်အလက်များအတွက် ပရော်ဖက်ရှင်နယ် လူသားဘာသာပြန်သူဝန်ဆောင်မှုကို အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုခြင်းမှ ဖြစ်ပေါ်လာသော နားလည်မှုကွာခြားမှုများ သို့မဟုတ် မမှန်ကန်သော အသုံးပြုမှုများအတွက် ကျွန်ုပ်တို့ တာဝန်မခံပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
